# LoRA training

Runtime → Change runtime type → **T4 GPU**. Add a `GITHUB_TOKEN` secret (key icon on the left, read-only access to the repo).

If Colab disconnects, run all cells again: training resumes from the last checkpoint on Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from google.colab import userdata

BRANCH = 'vito-lora-training'
REPO = '/content/spatial-probing'

token = userdata.get('GITHUB_TOKEN')
if not os.path.exists(REPO):
    !git clone -q -b {BRANCH} https://{token}@github.com/vitorizz/spatial-probing.git {REPO}
%cd {REPO}
!git checkout -q {BRANCH} && git pull -q
!git log --oneline -1

In [ ]:
# Keep Colab's CUDA torch: installing torch from requirements.txt replaces it with a CPU build.
!grep -v -E '^torch' requirements.txt > /tmp/req.txt && pip install -q -r /tmp/req.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Targeted layers. Runs Person A's pipeline only if selected_layers.json is not on this Drive yet (~15-20 min).
import json
SELECTED = '/content/drive/MyDrive/spatial_probe/results/probes/selected_layers.json'
if not os.path.exists(SELECTED):
    !python -m src.data.whatsup
    !python -m src.extract_hidden
    !python -m src.probe
    !python -m src.select_layers
sel = json.load(open(SELECTED))
print('case', sel['case'], '| S =', sel['S'], '| PEFT layers =', sel['peft_layers_to_transform'])
# Person A's frozen result (NOTES.md, 2026-11-09). If this fails, stop and check with A before training.
assert sel['peft_layers_to_transform'] == list(range(28, 36)), 'layer selection differs from A'

In [ ]:
CONFIG = 'configs/lora_targeted.yaml'
MAX_EXAMPLES = 200   # smoke test; set to None for the full run

extra = f'--max-examples {MAX_EXAMPLES}' if MAX_EXAMPLES else ''
!python -m src.train_lora --config {CONFIG} {extra}

In [ ]:
import yaml
run = yaml.safe_load(open(CONFIG))['run_name'] + (f'_smoke{MAX_EXAMPLES}' if MAX_EXAMPLES else '')
run_dir = f'/content/drive/MyDrive/spatial_probe/runs/{run}'
!cat {run_dir}/train_log.csv
!echo
!cat {run_dir}/val_curve.csv